# Exploratory Data Analysis - AG News (PRESS)

A quick look at the data we will train on: the AG News subset prepared by
`data/data_fetcher.py` and saved under `data/downloads/` as Parquet.

AG News is a topic-classification dataset. Each row is a short news snippet
(`text`) labelled with one of four balanced topics (`label`):

| label | class |
| --- | --- |
| 0 | World |
| 1 | Sports |
| 2 | Business |
| 3 | Sci/Tech |

The goal is to understand the data before modelling: sizes, class balance,
text lengths (which inform the tokenizer `max_length`), example rows, and basic
quality checks.

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 200)

DATA_DIR = Path("../data/downloads")
CLASS_NAMES = ["World", "Sports", "Business", "Sci/Tech"]  # label index -> name

train = pd.read_parquet(DATA_DIR / "train.parquet")
val = pd.read_parquet(DATA_DIR / "validation.parquet")

# Parquet stores `label` as an int; add a readable class column for the analysis.
for df in (train, val):
    df["class"] = df["label"].map(dict(enumerate(CLASS_NAMES)))

print(f"train: {len(train):,} rows | validation: {len(val):,} rows")
train.head(3)

## Schema & types

Both splits share the same two columns produced by the fetcher (`text`, `label`);
`class` is the readable label we just added.

In [ ]:
print("Columns:", list(train.columns))
train.dtypes

## Class distribution

AG News is balanced by design. We check that our random subset kept that balance.
If it did, plain **accuracy** is a meaningful metric (no single class dominates),
and we will still report **macro-F1** as a robustness check.

In [ ]:
dist = pd.DataFrame({
    "train": train["class"].value_counts(),
    "train_%": (train["class"].value_counts(normalize=True) * 100).round(1),
    "validation": val["class"].value_counts(),
    "validation_%": (val["class"].value_counts(normalize=True) * 100).round(1),
})
dist.loc[CLASS_NAMES]

## Text length

How long are the snippets? This directly informs the `max_length` used when
tokenizing for DistilBERT: if almost every text fits in N tokens, a larger
`max_length` only wastes CPU. Word count is a rough proxy for token count - the
real tokenizer splits into sub-words, so token counts run somewhat higher.

In [ ]:
train["n_chars"] = train["text"].str.len()
train["n_words"] = train["text"].str.split().str.len()

train[["n_chars", "n_words"]].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(1)

Average length per class, in case some topics are systematically longer.

In [ ]:
train.groupby("class")[["n_chars", "n_words"]].mean().round(1).loc[CLASS_NAMES]

## Example rows per class

A couple of real snippets per topic, to get a feel for the writing style.

In [ ]:
for name in CLASS_NAMES:
    print(f"\n=== {name} ===")
    for text in train.loc[train["class"] == name, "text"].head(2):
        print(" -", text)

## Data quality

Quick sanity checks: missing values and exact-duplicate texts.

In [ ]:
print("Missing values per column:")
print(train[["text", "label"]].isna().sum())
print(f"\nExact-duplicate texts in train: {train['text'].duplicated().sum()}")

## Takeaways

- **Balanced classes** -> accuracy (plus macro-F1) is a fair metric.
- **Short texts** -> a small tokenizer `max_length` (around 128) covers almost
  every row, keeping CPU training fast.
- **Clean data** -> no missing values to handle; exact duplicates, if any, are
  negligible for a toy dataset.

Next step: `src/data/dataset.py` turns this text into the token IDs DistilBERT
expects, using the `max_length` suggested above.